# 1.Task
Perform K-Prototypes clustering on the dataset "/content/df_final.csv" by loading the data, preprocessing it (cleaning, standardizing numerical features, and identifying categorical indices), running the algorithm for 2 to 10 clusters to calculate Cost, Calinski-Harabasz, and Davies-Bouldin scores, and visualizing these metrics to determine the optimal number of clusters.

## Load Libraries and Data

### Subtask:
Install necessary packages, import libraries, and load the dataset.


**Reasoning**:
Install the `kmodes` library as required for the K-Prototypes algorithm. This is a shell command and should be executed in its own cell.



In [ ]:
!pip install kmodes

**Reasoning**:
Import the necessary libraries and load the dataset into a DataFrame using the specified separator ';', then display the first few rows and the DataFrame info.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from kmodes.kprototypes import KPrototypes
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

# Carregar o conjunto de dados
df = pd.read_csv('/content/df_final.csv', sep=';')

# Exibir as primeiras linhas e informações
display(df.head())
print(df.info())

## Data Preprocessing

### Subtask:
Clean the data, sample it for performance, standardize numerical features, and prepare the dataset for K-Prototypes.


**Reasoning**:
Perform data cleaning, sampling, feature scaling, and preparation for K-Prototypes clustering as per the instructions.



In [ ]:
# 1. Descartar colunas e remover duplicatas
cols_to_drop = ['Unnamed: 0', 'num_acidente', 'data_acidente']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
df.drop_duplicates(inplace=True)

# 2. Descartar valores ausentes
df.dropna(inplace=True)

# 3. Criar uma amostra aleatória de 15.000 linhas
df_sample = df.sample(n=15000, random_state=42).reset_index(drop=True)

# 4. Identificar colunas numéricas e categóricas
numerical_cols = df_sample.select_dtypes(exclude='object').columns
categorical_cols = df_sample.select_dtypes(include='object').columns

# 5. Padronizar colunas numéricas
scaler = StandardScaler()
df_num_scaled = pd.DataFrame(scaler.fit_transform(df_sample[numerical_cols]), columns=numerical_cols)

# 6. Concatenar colunas numéricas padronizadas e categóricas
# Garantir que os índices sejam redefinidos para a concatenação correta
df_cat = df_sample[categorical_cols].reset_index(drop=True)
df_model = pd.concat([df_num_scaled, df_cat], axis=1)

# 7. Identificar índices inteiros das colunas categóricas
categorical_indices = [df_model.columns.get_loc(col) for col in categorical_cols]

# 8. Exibir resultados
display(df_model.head())
print(f"Categorical Indices: {categorical_indices}")

## Cluster Optimization Loop

### Subtask:
Run K-Prototypes for clusters 2 to 10 and compute Cost, Calinski-Harabasz, and Davies-Bouldin scores.


**Reasoning**:
Perform the K-Prototypes clustering optimization loop. First, create a dummy-encoded version of the dataset for calculating validation metrics (Calinski-Harabasz and Davies-Bouldin). Then, iterate through cluster counts from 2 to 10, fitting the model, storing the cost and metrics, and printing the results for each iteration.



In [ ]:
# 1. Preparar versão totalmente numérica para métricas
df_metrics = pd.get_dummies(df_model, drop_first=True)

# 2. Inicializar listas para armazenar resultados
costs = []
ch_scores = []
db_scores = []

# 3. Loop através de k de 2 a 10
for k in range(2, 11):
    try:
        # 4. Instanciar KPrototypes
        kp = KPrototypes(n_clusters=k, init='Cao', n_init=3, verbose=0, random_state=42)

        # 5. Ajustar o modelo
        # Converter para matriz é muitas vezes mais seguro para kmodes, mas passaremos df_model conforme solicitado
        # Se índices de colunas específicos forem passados, ele trata colunas por índice
        kp.fit(df_model, categorical=categorical_indices)

        # 6. Extrair rótulos e custo
        labels = kp.labels_
        cost = kp.cost_

        # 7. Anexar custo
        costs.append(cost)

        # 8. Calcular e anexar pontuação Calinski-Harabasz
        ch = calinski_harabasz_score(df_metrics, labels)
        ch_scores.append(ch)

        # 9. Calcular e anexar pontuação Davies-Bouldin
        db = davies_bouldin_score(df_metrics, labels)
        db_scores.append(db)

        # 10. Imprimir progresso
        print(f"k={k}: Cost={cost:.2f}, CH Score={ch:.2f}, DB Score={db:.2f}")

    except Exception as e:
        print(f"Erro para k={k}: {e}")

## Visualize Metrics

### Subtask:
Visualize the clustering performance metrics (Cost, Calinski-Harabasz, and Davies-Bouldin) using line plots to identify the optimal number of clusters.


**Reasoning**:
Visualize the clustering metrics (Cost, Calinski-Harabasz, and Davies-Bouldin scores) using line plots to help identify the optimal number of clusters.



In [ ]:
# Definir o intervalo de clusters
K = range(2, 11)

# Criar uma figura com 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Plotar Custo (Método do Cotovelo)
sns.lineplot(x=K, y=costs, marker='o', ax=axes[0], color='blue')
axes[0].set_title('Método do Cotovelo')
axes[0].set_xlabel('Número de Clusters (k)')
axes[0].set_ylabel('Custo')
axes[0].grid(True)

# Plotar Pontuação Calinski-Harabasz
sns.lineplot(x=K, y=ch_scores, marker='o', ax=axes[1], color='orange')
axes[1].set_title('Pontuação Calinski-Harabasz')
axes[1].set_xlabel('Número de Clusters (k)')
axes[1].set_ylabel('Pontuação')
axes[1].grid(True)

# Plotar Pontuação Davies-Bouldin
sns.lineplot(x=K, y=db_scores, marker='o', ax=axes[2], color='green')
axes[2].set_title('Pontuação Davies-Bouldin')
axes[2].set_xlabel('Número de Clusters (k)')
axes[2].set_ylabel('Pontuação')
axes[2].grid(True)

# Ajustar layout e exibir
plt.tight_layout()
plt.show()

## Final Task

### Subtask:
Summarize the preprocessing steps and the results of the clustering metrics.


## Summary:

### Data Analysis Key Findings

*   **Data Preparation and Sampling**: The original dataset contained **1,883,522 rows** and 61 columns. To optimize algorithm performance, the data was cleaned (dropping columns like `num_acidente` and `data_acidente`), duplicates were removed, and a random sample of **15,000 rows** was generated.
*   **Feature Engineering**: Numerical features were standardized using `StandardScaler`. The dataset was formatted for K-Prototypes by identifying **7 categorical columns** (located at indices 51 through 57) and concatenating them with the scaled numerical data.
*   **Clustering Cost**: The Cost metric demonstrated a consistent decrease as the number of clusters increased, dropping from approximately **641,331** at \$k=2\$ to **518,896** at \$k=10\$.
*   **Calinski-Harabasz (CH) Score**: The CH score trended downwards as clusters increased, starting at its highest point of **817.01** at \$k=2\$ and ending at **472.56** at \$k=10\$.
*   **Davies-Bouldin (DB) Score**: The DB score improved (decreased) as the number of clusters grew, moving from **3.39** at \$k=2\$ to its best value of **2.22** at \$k=10\$.

### Insights or Next Steps

*   **Conflicting Metric Signals**: The metrics suggest different optimal cluster counts; the CH score favors fewer clusters (\$k=2\$), while the DB score indicates better separation at higher cluster counts (\$k=10\$). A visual inspection of the "Elbow" in the Cost plot is required to balance these findings.
*   **Cluster Profiling**: Once the optimal \$k\$ is selected based on the visual plots, the next step is to fit the final model and analyze the cluster centroids (numerical means) and modes (categorical frequency) to interpret the specific characteristics of each accident group.


# 2. Task
Perform K-Prototypes clustering on the full dataset "/content/df_final.csv", removing the previous 15,000 sample limit to use the entire dataset. The process should include data preprocessing (cleaning, dropping duplicates and missing values, standardizing numerical features, and identifying categorical indices), running the K-Prototypes algorithm for clusters 2 to 10 to compute Cost, Calinski-Harabasz, and Davies-Bouldin scores, and visualizing these metrics to identify the optimal number of clusters, followed by a summary of the results.

## Data Preprocessing (Full Dataset)

### Subtask:
Clean the full dataset, standardize numerical features, and prepare it for K-Prototypes without sampling.


**Reasoning**:
Perform the requested data preprocessing on the full dataset: clean the data, standardize numerical features, prepare the dataframe for K-Prototypes, and identify categorical indices, as specified in the instructions.



In [ ]:
# 1. Limpar o DataFrame df (garantir robustez caso já esteja limpo)
cols_to_drop = ['Unnamed: 0', 'num_acidente', 'data_acidente']
existing_cols = [c for c in cols_to_drop if c in df.columns]
if existing_cols:
    df.drop(columns=existing_cols, inplace=True)

# 2. Remover duplicatas e valores ausentes
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

# 3. Identificar colunas numéricas e categóricas
numerical_cols = df.select_dtypes(exclude='object').columns
categorical_cols = df.select_dtypes(include='object').columns

# 4. Padronizar colunas numéricas (Full Dataset)
scaler_full = StandardScaler()
df_num_scaled_full = pd.DataFrame(scaler_full.fit_transform(df[numerical_cols]), columns=numerical_cols)

# 5. Concatenate numerical and categorical columns
# Reset indices to ensure correct alignment
df_cat_full = df[categorical_cols].reset_index(drop=True)
df_model_full = pd.concat([df_num_scaled_full, df_cat_full], axis=1)

# 6. Identify categorical indices
categorical_indices_full = [df_model_full.columns.get_loc(col) for col in categorical_cols]

# 7. Print results
print(f"Shape of df_model_full: {df_model_full.shape}")
print(f"Categorical Indices (Full): {categorical_indices_full}")
display(df_model_full.head())

## Cluster Optimization Loop

### Subtask:
Run K-Prototypes for clusters 2 to 10 on the full dataset and compute Cost, Calinski-Harabasz, and Davies-Bouldin scores.


**Reasoning**:
Perform the K-Prototypes clustering optimization loop on the full dataset. To prevent Out-Of-Memory errors due to high cardinality in categorical columns (e.g., 'municipio'), I will first attempt to create a dummy-encoded dataframe but fallback to using only numerical columns if the cardinality is too high. Then, I will iterate through cluster counts from 2 to 10, fitting the model, calculating the metrics, and printing the results.



In [ ]:
# 1. Prepare df_metrics_full with safety check for high cardinality
try:
    # Check estimated columns after dummy encoding
    # Heuristic: If categorical cardinality sum > 100, use numerical only to save memory on 1.5M rows
    total_cats = sum(df[col].nunique() for col in categorical_cols)
    if total_cats > 100:
        print(f"High cardinality detected ({total_cats} unique categories). Using numerical columns only for metrics to avoid OOM.")
        df_metrics_full = df_num_scaled_full.copy()
    else:
        df_metrics_full = pd.get_dummies(df_model_full, drop_first=True)
except Exception as e:
    print(f"Error preparing dummies: {e}. Fallback to numerical columns for metrics.")
    df_metrics_full = df_num_scaled_full.copy()

# 2. Initialize lists
costs_full = []
ch_scores_full = []
db_scores_full = []

# 3. Loop through k from 2 to 10
for k in range(2, 11):
    try:
        # 4. Instantiate KPrototypes
        # n_init=1 to reduce computation time on large dataset
        kp = KPrototypes(n_clusters=k, init='Cao', n_init=1, verbose=1, random_state=42)

        # 5. Fit the model
        kp.fit(df_model_full, categorical=categorical_indices_full)

        # 6. Retrieve cost and labels
        cost = kp.cost_
        labels = kp.labels_
        costs_full.append(cost)

        # 7. Calculate metrics
        ch = calinski_harabasz_score(df_metrics_full, labels)
        ch_scores_full.append(ch)

        db = davies_bouldin_score(df_metrics_full, labels)
        db_scores_full.append(db)

        # 8. Print progress
        print(f"k={k}: Cost={cost:.2f}, CH Score={ch:.2f}, DB Score={db:.2f}")

    except Exception as e:
        print(f"Error for k={k}: {e}")

**Reasoning**:
Visualize the clustering metrics (Cost, Calinski-Harabasz, and Davies-Bouldin scores) calculated on the full dataset to help identify the optimal number of clusters.



In [ ]:
# Define the range of clusters
K = range(2, 11)

# Create a figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Plot Cost (Elbow Method)
sns.lineplot(x=K, y=costs_full, marker='o', ax=axes[0], color='blue')
axes[0].set_title('Elbow Method (Full Dataset)')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Cost')
axes[0].grid(True)

# Plot Calinski-Harabasz Score
sns.lineplot(x=K, y=ch_scores_full, marker='o', ax=axes[1], color='orange')
axes[1].set_title('Calinski-Harabasz Score (Full Dataset)')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Score')
axes[1].grid(True)

# Plot Davies-Bouldin Score
sns.lineplot(x=K, y=db_scores_full, marker='o', ax=axes[2], color='green')
axes[2].set_title('Davies-Bouldin Score (Full Dataset)')
axes[2].set_xlabel('Number of Clusters (k)')
axes[2].set_ylabel('Score')
axes[2].grid(True)

# Adjust layout and display
plt.tight_layout()
plt.show()

## Final Task

### Subtask:
Summarize the preprocessing steps and the results of the clustering metrics on the full dataset.


## Summary:

### Q&A
**What is the optimal number of clusters for the full dataset?**
Based on the metrics computed, **6 clusters ($k=6$)** is the optimal choice. It achieved the best (lowest) Davies-Bouldin score of **1.80** and the second-highest Calinski-Harabasz score of **55,816**, indicating a strong balance between cluster compactness and separation. While $k=3$ had the highest Calinski-Harabasz score, $k=6$ provides a better structural definition according to the Davies-Bouldin index.

### Data Analysis Key Findings
*   **Dataset Magnitude**: The full dataset was successfully preprocessed, resulting in **1,589,417 rows** and **58 columns**. Seven categorical columns were identified for the K-Prototypes algorithm.
*   **High Cardinality Constraint**: The categorical features exhibited high cardinality (**64,919 unique categories**), necessitating a fallback to using only standardized numerical columns for calculating validation metrics (CH and DB scores) to prevent memory errors.
*   **Performance Metrics**:
    *   **Cost**: Decreased consistently as $k$ increased, dropping from **82.05M** ($k=2$) to **66.34M** ($k=10$).
    *   **Calinski-Harabasz (CH)**: Peaked at **$k=3$** (**56,399**) but showed a very strong secondary peak at **$k=6$** (**55,816**).
    *   **Davies-Bouldin (DB)**: The best score was observed at **$k=6$** (**1.80**), significantly better than $k=3$ (2.55) and $k=2$ (3.09).

### Insights or Next Steps
*   **Selection of k=6**: The analysis strongly suggests proceeding with **6 clusters**. Although $k=3$ separates data well (high CH), $k=6$ offers tighter clusters (low DB) which is often more useful for granular segmentation in large datasets.
*   **Next Step**: Train the final K-Prototypes model with $k=6$ on the full dataset and analyze the centroids (prototypes) to interpret the specific characteristics of each accident profile.


# Task
Train the final K-Prototypes model with `n_clusters=6` on the full preprocessed dataset (`df_model_full`) using the identified categorical indices. Assign the cluster labels to the original dataframe (`df`) as a new column named `cluster`. Analyze the results by visualizing the cluster counts with a bar chart and displaying the cluster centroids to define the profile of each group, followed by a summary of the final model application and cluster distribution.

## Train Final K-Prototypes Model

### Subtask:
Train the K-Prototypes model with the optimal number of clusters (k=6) on the full dataset.


**Reasoning**:
Train the final K-Prototypes model with k=6 on the full dataset as determined by the previous analysis.



In [ ]:
# Instantiate KPrototypes with optimal k=6
kp_final = KPrototypes(n_clusters=6, init='Cao', n_init=1, verbose=1, random_state=42)

# Fit the model to the full dataset
print("Starting training for k=6...")
kp_final.fit(df_model_full, categorical=categorical_indices_full)

# Display completion message and final cost
print(f"Training complete. Final Cost: {kp_final.cost_:.2f}")

## Assign Cluster Labels and Cluster Analysis

### Subtask:
Assign the generated cluster labels to the original dataframe, visualize the cluster distribution, and analyze the cluster centroids to define the profiles.


**Reasoning**:
Assign the cluster labels to the original dataframe, visualize the distribution, and interpret the centroids by reverting the scaling on numerical features and displaying the categorical modes.



In [ ]:
# 1. Atribuir rótulos ao DataFrame original
# Resetar o índice para garantir alinhamento com os rótulos gerados (que seguem a ordem posicional)
df.reset_index(drop=True, inplace=True)
df['cluster'] = kp_final.labels_

# 2. Visualizar distribuição dos clusters
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='cluster', palette='viridis', hue='cluster', legend=False)
plt.title('Distribuição dos Clusters de Acidentes (Dataset Completo)')
plt.xlabel('Cluster')
plt.ylabel('Quantidade de Acidentes')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 3. Extrair centroides
# kp_final.cluster_centroids_ retorna uma lista: [0] = numérico, [1] = categórico
centroids_num_scaled = kp_final.cluster_centroids_[0]
centroids_cat = kp_final.cluster_centroids_[1]

# 4. Criar DataFrame para centroides numéricos e reverter escala
# Reverter a padronização para interpretar os valores reais (médias)
centroids_num_original = scaler_full.inverse_transform(centroids_num_scaled)
df_centroids_num = pd.DataFrame(centroids_num_original, columns=numerical_cols)
df_centroids_num['cluster'] = range(len(df_centroids_num))

# 5. Criar DataFrame para centroides categóricos (modas)
df_centroids_cat = pd.DataFrame(centroids_cat, columns=categorical_cols)
df_centroids_cat['cluster'] = range(len(df_centroids_cat))

# 6. Exibir os resultados para análise
print("Perfil dos Clusters - Atributos Numéricos (Médias):")
display(df_centroids_num.set_index('cluster'))

print("\nPerfil dos Clusters - Atributos Categóricos (Modas):")
display(df_centroids_cat.set_index('cluster'))

**Reasoning**:
The previous execution failed because `kp_final.cluster_centroids_` returned a single numpy array of mixed types (strings) instead of a list of two arrays (numerical and categorical), causing the categorical values (like 'SEXTA-FEIRA') to be included in the array passed to `inverse_transform`. To fix this, I will check the structure of `cluster_centroids_` and, if it is a single array, slice it based on the number of numerical columns to separate the numerical and categorical centroids correctly. I will also ensure the numerical part is converted to float before inverse transformation.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Atribuir rótulos ao DataFrame original
df.reset_index(drop=True, inplace=True)
df['cluster'] = kp_final.labels_

# 2. Visualizar distribuição dos clusters
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='cluster', palette='viridis', hue='cluster', legend=False)
plt.title('Distribuição dos Clusters de Acidentes (Dataset Completo)')
plt.xlabel('Cluster')
plt.ylabel('Quantidade de Acidentes')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 3. Extrair e separar centroides
# Verificar se cluster_centroids_ é uma lista (split) ou um array único (mixed)
centroids = kp_final.cluster_centroids_
num_feat_count = len(numerical_cols)

if isinstance(centroids, list):
    # Formato padrão kprototypes: lista [num, cat]
    centroids_num_scaled = np.array(centroids[0], dtype=float)
    centroids_cat = np.array(centroids[1])
else:
    # Formato array único (comum quando tipos mistos forçam dtype=object)
    # Assumindo que a ordem das colunas segue df_model_full (Numéricas + Categóricas)
    print("Detectado array único de centroides. Separando colunas...")
    centroids_num_scaled = np.array(centroids[:, :num_feat_count], dtype=float)
    centroids_cat = np.array(centroids[:, num_feat_count:])

# 4. Criar DataFrame para centroides numéricos e reverter escala
centroids_num_original = scaler_full.inverse_transform(centroids_num_scaled)
df_centroids_num = pd.DataFrame(centroids_num_original, columns=numerical_cols)
df_centroids_num['cluster'] = range(len(df_centroids_num))

# 5. Criar DataFrame para centroides categóricos (modas)
df_centroids_cat = pd.DataFrame(centroids_cat, columns=categorical_cols)
df_centroids_cat['cluster'] = range(len(df_centroids_cat))

# 6. Exibir os resultados para análise
print("Perfil dos Clusters - Atributos Numéricos (Médias):")
display(df_centroids_num.set_index('cluster'))

print("\nPerfil dos Clusters - Atributos Categóricos (Modas):")
display(df_centroids_cat.set_index('cluster'))

In [ ]:
df.to_csv("df_cluster.csv",sep=";")